In [1]:
!pip install datasets -q

import sys
sys.path.append('/kaggle/working')

In [2]:
# invariant-core/data/loaders/xnli_loader.py

from datasets import load_dataset
import pandas as pd
from collections import defaultdict

# Наши 8 сред: регион → языки XNLI
REGION_LANGUAGES = {
    "eastern_europe": ["ru", "bg"],        # болгарский есть в XNLI
    "western_europe": ["de", "fr", "es"],
    "anglosphere":    ["en"],
    "arab_world":     ["ar"],
    "south_asia":     ["hi", "ur"],
    "east_asia":      ["zh"],
    "southeast_asia": ["th", "sw"],        # суахили как прокси Африки
    "latin_america":  ["es"],              # дублирует, но с другим split
}

# XNLI содержит 15 языков: ar, bg, de, el, en, es, fr, hi,
# ru, sw, th, tr, ur, vi, zh

def load_xnli_by_region(split="validation", samples_per_lang=5000):
    """
    Загружает XNLI, группирует по регионам-средам.
    split: "train" | "validation" | "test"
    samples_per_lang: сколько примеров брать на язык
    """

    # XNLI загружается отдельно на каждый язык
    region_data = defaultdict(list)
    lang_stats = {}

    for region, languages in REGION_LANGUAGES.items():
        print(f"\n→ Загрузка региона: {region}")

        for lang in languages:
            try:
                # Загружаем датасет для языка
                dataset = load_dataset(
                    "xnli",
                    lang,
                    split=split,
                )

                # Берём нужное количество
                n = min(samples_per_lang, len(dataset))
                subset = dataset.select(range(n))

                # Добавляем метаданные
                for item in subset:
                    region_data[region].append({
                        "premise":    item["premise"],
                        "hypothesis": item["hypothesis"],
                        "label":      item["label"],  # 0=entailment, 1=neutral, 2=contradiction
                        "language":   lang,
                        "region":     region,
                        # Позже добавим: source_type, time_period
                    })

                lang_stats[lang] = n
                print(f"  ✓ {lang}: {n} примеров")

            except Exception as e:
                print(f"  ✗ {lang}: ошибка — {e}")

    return region_data, lang_stats


def region_data_to_df(region_data):
    """Конвертирует в DataFrame для удобного анализа."""
    all_rows = []
    for region, items in region_data.items():
        all_rows.extend(items)
    return pd.DataFrame(all_rows)


def print_stats(df):
    """Базовая статистика по средам."""
    print("\n=== СТАТИСТИКА ПО СРЕДАМ ===")
    print(df.groupby("region")["label"].value_counts().unstack(fill_value=0))
    print(f"\nВсего примеров: {len(df)}")
    print(f"Языков: {df['language'].nunique()}")
    print(f"Регионов: {df['region'].nunique()}")

    # Баланс — важно для IRM
    sizes = df.groupby("region").size()
    print(f"\nРазмеры сред (мин/макс): {sizes.min()} / {sizes.max()}")
    ratio = sizes.max() / sizes.min()
    if ratio > 3:
        print(f"⚠ Дисбаланс {ratio:.1f}x — нужна стратификация")
    else:
        print(f"✓ Баланс в норме ({ratio:.1f}x)")


if __name__ == "__main__":
    print("Загружаем XNLI validation split...")
    region_data, lang_stats = load_xnli_by_region(
        split="validation",
        samples_per_lang=5000
    )

    df = region_data_to_df(region_data)
    print_stats(df)

    # Сохраняем
    df.to_parquet("xnli_by_region.parquet", index=False)
    print("\n✓ Сохранено: xnli_by_region.parquet")

Загружаем XNLI validation split...

→ Загрузка региона: eastern_europe


README.md: 0.00B [00:00, ?B/s]

ru/train-00000-of-00001.parquet:   0%|          | 0.00/70.0M [00:00<?, ?B/s]

ru/test-00000-of-00001.parquet:   0%|          | 0.00/477k [00:00<?, ?B/s]

ru/validation-00000-of-00001.parquet:   0%|          | 0.00/239k [00:00<?, ?B/s]

...

sw/validation-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

  ✓ sw: 2490 примеров

→ Загрузка региона: latin_america
  ✓ es: 2490 примеров

=== СТАТИСТИКА ПО СРЕДАМ ===
label              0     1     2
region                          
anglosphere      830   830   830
arab_world       830   830   830
east_asia        830   830   830
eastern_europe  1660  1660  1660
latin_america    830   830   830
south_asia      1660  1660  1660
southeast_asia  1660  1660  1660
western_europe  2490  2490  2490

Всего примеров: 32370
Языков: 12
Регионов: 8

Размеры сред (мин/макс): 2490 / 7470
✓ Баланс в норме (3.0x)

✓ Сохранено: xnli_by_region.parquet


In [3]:
region_data, stats = load_xnli_by_region(split="validation", samples_per_lang=2000)
df = region_data_to_df(region_data)
print_stats(df)

# Посмотреть несколько примеров
df.sample(5)


→ Загрузка региона: eastern_europe
  ✓ ru: 2000 примеров
  ✓ bg: 2000 примеров

→ Загрузка региона: western_europe
  ✓ de: 2000 примеров
  ✓ fr: 2000 примеров
  ✓ es: 2000 примеров

→ Загрузка региона: anglosphere
  ✓ en: 2000 примеров

→ Загрузка региона: arab_world
  ✓ ar: 2000 примеров

→ Загрузка региона: south_asia
  ✓ hi: 2000 примеров
  ✓ ur: 2000 примеров

→ Загрузка региона: east_asia
  ✓ zh: 2000 примеров

→ Загрузка региона: southeast_asia
  ✓ th: 2000 примеров
  ✓ sw: 2000 примеров

→ Загрузка региона: latin_america
  ✓ es: 2000 примеров

=== СТАТИСТИКА ПО СРЕДАМ ===
label              0     1     2
region                          
anglosphere      666   667   667
arab_world       666   667   667
east_asia        666   667   667
eastern_europe  1332  1334  1334
latin_america    666   667   667
south_asia      1332  1334  1334
southeast_asia  1332  1334  1334
western_europe  1998  2001  2001

Всего примеров: 26000
Языков: 12
Регионов: 8

Размеры сред (мин/макс): 2000 / 6000

,premise,hypothesis,label,language,region
4745,"Was wir sehen, sind Details.",Die Details sind sichtbar.,0,de,western_europe
17533,ان دستورالعمل اور اسباب کے مختصر خلاصہ کے لیے ...,دستخط مکمل طور پر سنبھالے گئے تھے.,2,ur,south_asia
14139,"मं , और वो ऐसे ही नगर छोड़ दिए .और वो, उसने अपन...",Usne apni behen ko uske baad kabi nahi dekha,0,hi,south_asia
2342,"добре защо не започнете, защото сте имали пове...",Защо не обясните книгата на всички ни.,1,bg,eastern_europe
16706,واضح طور پر آج کے بچے ٹی وی سیٹ کے سامنے بہت س...,"ٹی وی ابھی تک ایجاد نہیں کیا گیا ہے,لہذا زیادہ...",2,ur,south_asia


In [4]:
# Ячейка 4 — детектор противоречий

def extract_contradictions(df):
    """
    Выделяем contradiction-пары (label=2) по каждому региону.
    Это основа для оси декларация vs реальность.
    """
    contradictions = df[df["label"] == 2].copy()
    
    print("=== ПРОТИВОРЕЧИЯ ПО РЕГИОНАМ ===")
    stats = contradictions.groupby("region").size()
    print(stats)
    print(f"\nВсего contradiction-пар: {len(contradictions)}")
    print(f"Это {len(contradictions)/len(df)*100:.1f}% от всех данных")
    
    # Смотрим примеры из каждого региона
    print("\n=== ПРИМЕРЫ ПРОТИВОРЕЧИЙ ===")
    for region in df["region"].unique():
        sample = contradictions[contradictions["region"] == region].sample(1).iloc[0]
        print(f"\n[{region} / {sample['language']}]")
        print(f"  Утверждение: {sample['premise'][:100]}...")
        print(f"  Противоречие: {sample['hypothesis'][:100]}...")
    
    return contradictions

contradictions = extract_contradictions(df)

# Сохраняем отдельно
contradictions.to_parquet("contradictions_by_region.parquet", index=False)
print("\n✓ Сохранено: contradictions_by_region.parquet")

=== ПРОТИВОРЕЧИЯ ПО РЕГИОНАМ ===
region
anglosphere        667
arab_world         667
east_asia          667
eastern_europe    1334
latin_america      667
south_asia        1334
southeast_asia    1334
western_europe    2001
dtype: int64

Всего contradiction-пар: 8671
Это 33.4% от всех данных

=== ПРИМЕРЫ ПРОТИВОРЕЧИЙ ===

[eastern_europe / bg]
  Утверждение: Съквартирантът на Хазми си спомня как по това време е направил необяснимо пътуване до летището в Сан...
  Противоречие: Съквартирантът на Хазми никога не е чувал за летището в Сан Диего....

[western_europe / de]
  Утверждение: Die Effekte der Sozialhilfe auf Versender, welche wechseln, wurden genau so berechnet wie der obenst...
  Противоречие: Sie wissen nicht, wie Effekte der Sozialhilfe berechnet werden sollen....

[anglosphere / en]
  Утверждение: But I do not believe any such algorithmic tool can be complete....
  Противоречие: These tools work flawlessly under any circumstances....

[arab_world / ar]
  Утверждение: ستتاح الف

In [5]:
# Ячейка 5 — расширенные маркеры

import re

DECLARATION_MARKERS = {
    "ru": ["должен", "нужно", "следует", "считаю", "полагаю", "верю",
           "мы всегда", "принято", "нельзя", "обязан", "важно", "необходимо",
           "правильно", "неправильно", "хорошо", "плохо", "лучше", "хуже",
           "надо", "стоит", "желательно", "запрещено", "разрешено"],
    "bg": ["трябва", "вярвам", "винаги", "никога", "трябва да", "важно",
           "необходимо", "правилно", "погрешно"],
    "en": ["should", "must", "believe", "think", "always", "never",
           "supposed to", "ought", "consider", "feel that", "important",
           "necessary", "right", "wrong", "better", "worse", "need to",
           "have to", "it is good", "it is bad", "we should", "one must"],
    "de": ["sollte", "muss", "glaube", "denke", "immer", "nie", "wichtig",
           "notwendig", "richtig", "falsch", "besser", "schlechter", "müssen",
           "dürfen", "sollen"],
    "fr": ["devrait", "doit", "crois", "pense", "toujours", "jamais",
           "faut", "important", "nécessaire", "correct", "incorrect",
           "mieux", "pire", "on doit", "il faut"],
    "es": ["debe", "hay que", "creo", "pienso", "siempre", "nunca",
           "debería", "importante", "necesario", "correcto", "incorrecto",
           "mejor", "peor", "se debe", "es necesario", "es importante"],
    "ar": ["يجب", "ينبغي", "أعتقد", "دائما", "أبدا", "مهم", "ضروري",
           "صحيح", "خاطئ", "أفضل", "أسوأ", "لا بد", "من المهم"],
    "hi": ["चाहिए", "मानता", "हमेशा", "कभी नहीं", "जरूरी", "महत्वपूर्ण",
           "सही", "गलत", "बेहतर", "बुरा", "होना चाहिए"],
    "ur": ["چاہیے", "مانتا", "ہمیشہ", "کبھی نہیں", "ضروری", "اہم",
           "صحیح", "غلط", "بہتر", "برا"],
    "zh": ["应该", "必须", "认为", "总是", "从不", "重要", "必要",
           "正确", "错误", "更好", "更坏", "需要", "应当"],
    "th": ["ควร", "ต้อง", "เชื่อ", "คิด", "เสมอ", "ไม่เคย", "สำคัญ",
           "จำเป็น", "ถูก", "ผิด", "ดีกว่า", "แย่กว่า"],
    "sw": ["lazima", "inabidi", "nadhani", "daima", "kamwe", "muhimu",
           "sahihi", "vibaya", "bora", "mbaya zaidi", "inapaswa"],
}

BEHAVIOR_MARKERS = {
    "ru": ["сделал", "произошло", "составило", "было", "стало", "в результате",
           "по данным", "согласно", "статистика", "исследование показало",
           "выяснилось", "обнаружено", "зафиксировано", "процент", "число",
           "количество", "увеличилось", "уменьшилось", "погиб", "арестован"],
    "bg": ["се случи", "стана", "беше", "според", "статистика", "процент",
           "брой", "увеличи", "намали"],
    "en": ["happened", "occurred", "resulted", "according to", "statistics",
           "data shows", "was found", "percent", "number of", "increased",
           "decreased", "was arrested", "was killed", "study found",
           "research shows", "survey", "poll", "reported", "confirmed"],
    "de": ["geschah", "ergab", "laut", "Statistik", "Prozent", "Anzahl",
           "stieg", "sank", "wurde verhaftet", "Studie", "Forschung"],
    "fr": ["s'est passé", "selon", "statistiques", "résultat", "données",
           "pourcentage", "nombre", "augmenté", "diminué", "étude", "rapport"],
    "es": ["ocurrió", "según", "estadísticas", "resultado", "datos",
           "porcentaje", "número", "aumentó", "disminuyó", "estudio",
           "informe", "fue arrestado", "murió"],
    "ar": ["حدث", "وفقا", "إحصائيات", "نتيجة", "بيانات", "نسبة",
           "عدد", "ازداد", "انخفض", "دراسة", "تقرير"],
    "hi": ["हुआ", "के अनुसार", "आंकड़े", "परिणाम", "प्रतिशत", "संख्या",
           "बढ़ा", "घटा", "अध्ययन", "रिपोर्ट"],
    "ur": ["ہوا", "کے مطابق", "اعداد", "نتیجہ", "فیصد", "تعداد",
           "بڑھا", "کم ہوا", "تحقیق", "رپورٹ"],
    "zh": ["发生", "根据", "统计", "结果", "数据", "百分", "数量",
           "增加", "减少", "研究", "报告", "被捕", "死亡"],
    "th": ["เกิดขึ้น", "ตาม", "สถิติ", "ผล", "ข้อมูล", "ร้อยละ",
           "จำนวน", "เพิ่มขึ้น", "ลดลง", "การศึกษา", "รายงาน"],
    "sw": ["ilitokea", "kulingana", "takwimu", "matokeo", "data",
           "asilimia", "idadi", "iliongezeka", "ilipungua", "utafiti",
           "ripoti", "alikamatwa"],
}

def detect_source_type(text, language):
    text_lower = text.lower()
    decl_markers  = DECLARATION_MARKERS.get(language, DECLARATION_MARKERS["en"])
    behav_markers = BEHAVIOR_MARKERS.get(language, BEHAVIOR_MARKERS["en"])
    decl_score  = sum(1 for m in decl_markers  if m in text_lower)
    behav_score = sum(1 for m in behav_markers if m in text_lower)
    if decl_score > behav_score:
        return "declaration"
    elif behav_score > decl_score:
        return "behavior"
    else:
        return "neutral"

def add_source_type(df):
    df = df.copy()
    df["source_type"] = df.apply(
        lambda row: detect_source_type(row["premise"], row["language"]),
        axis=1
    )
    return df

# Загружаем сохранённый файл (чтобы не перекачивать XNLI)
df = pd.read_parquet("xnli_by_region.parquet")
df = add_source_type(df)

print("=== ТИПЫ ИСТОЧНИКОВ ПО РЕГИОНАМ ===")
print(df.groupby(["region", "source_type"]).size().unstack(fill_value=0))
print("\n=== ОБЩЕЕ РАСПРЕДЕЛЕНИЕ ===")
print(df["source_type"].value_counts())
print(f"\nПроцент размеченных (не neutral): "
      f"{(df['source_type'] != 'neutral').mean()*100:.1f}%")

df.to_parquet("xnli_with_source_type.parquet", index=False)
print("\n✓ Сохранено: xnli_with_source_type.parquet")

=== ТИПЫ ИСТОЧНИКОВ ПО РЕГИОНАМ ===
source_type     behavior  declaration  neutral
region                                        
anglosphere           60          348     2082
arab_world           162          210     2118
east_asia            141          261     2088
eastern_europe       408          378     4194
latin_america         84          273     2133
south_asia           369          501     4110
southeast_asia       498          777     3705
western_europe       234         1014     6222

=== ОБЩЕЕ РАСПРЕДЕЛЕНИЕ ===
source_type
neutral        26652
declaration     3762
behavior        1956
Name: count, dtype: int64

Процент размеченных (не neutral): 17.7%

✓ Сохранено: xnli_with_source_type.parquet


In [6]:
# Ячейка 6 — финальная проверка фазы 0

df = pd.read_parquet("xnli_with_source_type.parquet")

print("=== ФИНАЛ ФАЗЫ 0 — ИТОГОВЫЙ ДАТАСЕТ ===\n")
print(f"Примеров всего:        {len(df)}")
print(f"Регионов (сред):       {df['region'].nunique()}")
print(f"Языков:                {df['language'].nunique()}")
print(f"Меток (0/1/2):         равномерно ✓")
print(f"Размечено по типу:     {(df['source_type'] != 'neutral').mean()*100:.1f}%")

print("\n=== ПРИМЕР DECLARATION ===")
row = df[df["source_type"] == "declaration"].sample(1).iloc[0]
print(f"Язык: {row['language']} | Регион: {row['region']}")
print(f"Текст: {row['premise'][:200]}")

print("\n=== ПРИМЕР BEHAVIOR ===")
row = df[df["source_type"] == "behavior"].sample(1).iloc[0]
print(f"Язык: {row['language']} | Регион: {row['region']}")
print(f"Текст: {row['premise'][:200]}")

print("\n=== КОЛОНКИ ДАТАСЕТА ===")
print(df.dtypes)
print("\n✓ Фаза 0 завершена. Готово к фазе 1 — baseline обучение.")

=== ФИНАЛ ФАЗЫ 0 — ИТОГОВЫЙ ДАТАСЕТ ===

Примеров всего:        32370
Регионов (сред):       8
Языков:                12
Меток (0/1/2):         равномерно ✓
Размечено по типу:     17.7%

=== ПРИМЕР DECLARATION ===
Язык: zh | Регион: east_asia
Текст: 第三个特征是状态空间中流动的趋同与发散，表征有序与混沌状态的特征，这对我们未来的讨论可能是最重要的。

=== ПРИМЕР BEHAVIOR ===
Язык: ur | Регион: south_asia
Текст: کیا یہ بیس فیصد سود ہے؟

=== КОЛОНКИ ДАТАСЕТА ===
premise        object
hypothesis     object
label           int64
language       object
region         object
source_type    object
dtype: object

✓ Фаза 0 завершена. Готово к фазе 1 — baseline обучение.
